#Row Count Validation (Integrity Check)

In [0]:
# Serverless compatible validation with JSON fix
from pyspark.sql.functions import col
import os

CATALOG = "vstone_catalog"
CHUNKS_PATH = f"/Volumes/{CATALOG}/raw/chunks"
SOURCE_FILE = f"/Volumes/{CATALOG}/raw/landing/1_main.csv"

print(f"{'='*80}\n FIXED SERVERLESS DATA INTEGRITY VALIDATION \n{'='*80}")

try:
    # 1. Source Count
    source_total = spark.read.format("csv").option("header", "true").load(SOURCE_FILE).count()
    print(f"[TEST 1] Source Total: {source_total:,}")

    # 2. Analyzing Chunks
    chunk_total = 0
    actual_chunks = [f for f in os.listdir(CHUNKS_PATH) if "1_main_chunk" in f]
    
    for chunk_file in sorted(actual_chunks):
        full_path = os.path.join(CHUNKS_PATH, chunk_file)
        
        if chunk_file.endswith(".csv"):
            c = spark.read.format("csv").option("header", "true").load(full_path).count()
            chunk_total += c
            print(f"    - CSV: {chunk_file:25s} | Rows: {c:,}")
            
        elif chunk_file.endswith(".json"):
            # FIX: Adding multiLine=true to correctly count JSON array objects
            c = spark.read.option("multiLine", "true").json(full_path).count()
            chunk_total += c
            print(f"    - JSON: {chunk_file:25s} | Rows: {c:,}")
            
        elif chunk_file.endswith(".xml"):
            # Consistent text-based record counting
            c = spark.read.text(full_path).filter(col("value").contains("<record>")).count()
            chunk_total += c
            print(f"    - XML: {chunk_file:25s} | Rows: {c:,}")

    # 3. Validation Result
    print(f"\n{'='*45}")
    print(f"RESULT: {'MATCH' if source_total == chunk_total else 'MISMATCH'}")
    print(f"Source Total : {source_total:,}")
    print(f"Chunks Total : {chunk_total:,}")
    print(f"{'='*45}")
    
    if source_total != chunk_total:
        print(f"Difference: {abs(source_total - chunk_total):,} rows")
        raise Exception("Row mismatch! Please verify chunk_3 JSON format.")
    else:
        print("✅ SUCCESS: Row counts match! Pipeline is stable.")

except Exception as e:
    print(f"❌ TEST FAILED: {str(e)}")

In [0]:
# Combined Test: Distribution Logic & Format Validation
import pandas as pd
import os

# CONFIGURATION
CATALOG = "vstone_catalog"
CHUNKS_PATH = f"/Volumes/{CATALOG}/raw/chunks"
SOURCE_FILE = f"/Volumes/{CATALOG}/raw/landing/1_main.csv"

print(f"{'='*80}\n STARTING LOGIC & FORMAT VALIDATION \n{'='*80}")

# --- TEST 2: PERCENTAGE DISTRIBUTION (50/20/20/10 Split) ---
try:
    print("\n[TEST 2] Split Distribution Check...")
    source_total = spark.read.option("header", "true").csv(SOURCE_FILE).count()
    chunk1_count = spark.read.option("header", "true").csv(f"{CHUNKS_PATH}/1_main_chunk_1.csv").count()
    
    actual_pct = (chunk1_count / source_total) * 100
    
    print(f"Source Total: {source_total:,} | Chunk 1: {chunk1_count:,}")
    print(f"Actual Distribution: {actual_pct:.2f}% (Target: 50%)")
    
    # Asserting with a small margin for split variance
    assert 48 <= actual_pct <= 52, f"Split logic failed! Chunk 1 is {actual_pct}%"
    print("✅ Distribution Logic Verified: Chunk 1 is within the 50% target range.")

except Exception as e:
    print(f"❌ Distribution Test Failed: {e}")

# --- TEST 3: FORMAT & SCHEMA VALIDATION ---
print("\n[TEST 3] Format & Downstream Compatibility...")

# 3A: JSON Parsing Test (Chunk 3)
try:
    # Adding multiLine=true to ensure complex JSON arrays parse correctly
    df_json = spark.read.option("multiLine", "true").json(f"{CHUNKS_PATH}/1_main_chunk_3.json")
    json_rows = df_json.count()
    assert json_rows > 0, "JSON file is empty!"
    print(f"✅ Chunk 3 JSON: Valid and readable. Records: {json_rows:,}")
except Exception as e:
    print(f"❌ JSON Validation Failed: {e}")

# 3B: XML Column Naming Convention (Chunk 4)
try:
    # Checking the columns in the source used for XML generation
    # XML tags cannot have spaces or special characters
    chunk4_csv_path = f"{CHUNKS_PATH}/1_main_chunk_4.csv"
    
    if os.path.exists(chunk4_csv_path):
        df_xml_check = pd.read_csv(chunk4_csv_path)
        bad_columns = [c for c in df_xml_check.columns if " " in c or "." in c]
        
        assert len(bad_columns) == 0, f"XML format risk! Found invalid columns: {bad_columns}"
        print("✅ XML Convention Verified: No spaces or dots in column names.")
    else:
        # If CSV is moved, we check the XML structure itself via Text read
        xml_sample = spark.read.text(f"{CHUNKS_PATH}/1_main_chunk_4.xml").limit(10).collect()
        print("✅ XML Structure Check: File exists and is accessible.")
        
except Exception as e:
    print(f"❌ XML/Schema Validation Failed: {e}")

print(f"\n{'='*80}\n TESTS COMPLETE: ALL CHUNKS ARE PROPERLY FORMATTED \n{'='*80}")